# Trabalho IAA011 — Visão Computacional

## Exercício 1: extração de características

Os bancos de imagens fornecidos são conjuntos de imagens de 250x250 pixels de imuno-histoquímica 
(biópsia) de câncer de mama. No total são 4 classes (0, 1+, 2+ e 3+) que estão divididas em diretórios. O 
objetivo é classificar as imagens nas categorias correspondentes. Uma base de imagens será utilizada para 
o treinamento e outra para o teste do treino.

### Passo 1 — Importações e configurações iniciais


In [29]:
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from skimage.feature import local_binary_pattern
from sklearn.model_selection import StratifiedGroupKFold
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.models import Model

### Configuração de pastas e parâmetros

Execute as células em ordem, com a pasta de trabalho do kernel definida como **Trabalho Imagens - IAA**, onde estão este notebook e as pastas `Train_Warwick` e `Test_Warwick`.

Os caminhos abaixo usam `Path / subpasta` para evitar os avisos de barras invertidas. A pasta de execução será exibida para facilitar a conferência.

In [30]:
# Execute o notebook com a pasta de trabalho "Trabalho Imagens - IAA".
BASE_DIR = Path.cwd()
TRAIN_DIR = BASE_DIR / "Train_Warwick" / "Train_4cls_amostra"
TEST_DIR = BASE_DIR / "Test_Warwick" / "Test_4cl_amostra"
OUTPUT_DIR = BASE_DIR / "saidas_exercicio1"

print("Pasta de execução:", BASE_DIR)
for pasta in (TRAIN_DIR, TEST_DIR):
    if not pasta.is_dir():
        raise FileNotFoundError(
            f"Pasta não encontrada: {pasta}. "
            "Configure a pasta de trabalho do kernel como 'Trabalho Imagens - IAA'."
        )
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Extensões consideradas como imagens.
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}

# Configuração LBP usada neste trabalho: 8 vizinhos e raio 1.
LBP_P = 8
LBP_R = 1
LBP_METHOD = "uniform"
N_LBP_BINS = LBP_P + 2  # para métod uniform, são P + 2 possíveis códigos

print("Pasta treino existe:", TRAIN_DIR.exists(), "->", TRAIN_DIR)
print("Pasta teste existe:", TEST_DIR.exists(), "->", TEST_DIR)

Pasta de execução: C:\workspace\ufpr\IAA-2026\TRABALHO_DE_IAA011\Trabalho Imagens - IAA
Pasta treino existe: True -> C:\workspace\ufpr\IAA-2026\TRABALHO_DE_IAA011\Trabalho Imagens - IAA\Train_Warwick\Train_4cls_amostra
Pasta teste existe: True -> C:\workspace\ufpr\IAA-2026\TRABALHO_DE_IAA011\Trabalho Imagens - IAA\Test_Warwick\Test_4cl_amostra


### Passo 2 — Carregar as imagens e identificar os pacientes

A função percorre as subpastas das classes e cria uma tabela com o caminho, nome do arquivo, identificador do paciente e rótulo. O ID é o texto antes do primeiro `_`, por exemplo, `01` em `01_HER2_0001.png`. Isso permite separar os pacientes sem misturar suas imagens.

In [31]:
def obter_id_paciente(nome_arquivo):
    partes = Path(nome_arquivo).stem.split("_")
    if len(partes) < 3:
        raise ValueError(f"Nome de arquivo fora do padrão esperado: {nome_arquivo}")
    return partes[0]

def carregar_manifesto(pasta_base):
    if not pasta_base.is_dir():
        raise FileNotFoundError(f"Pasta não encontrada: {pasta_base.resolve()}")

    linhas = []
    for pasta_classe in sorted(p for p in pasta_base.iterdir() if p.is_dir()):
        for arquivo in sorted(pasta_classe.rglob("*")):
            if arquivo.is_file() and arquivo.suffix.lower() in IMAGE_EXTENSIONS:
                linhas.append({
                    "image_path": str(arquivo.resolve()),
                    "filename": arquivo.name,
                    "patient_id": obter_id_paciente(arquivo.name),
                    "label": pasta_classe.name,
                })

    if not linhas:
        raise ValueError(f"Nenhuma imagem encontrada em {pasta_base}")
    return pd.DataFrame(linhas)


df_treino_original = carregar_manifesto(TRAIN_DIR)
df_teste = carregar_manifesto(TEST_DIR)

print("Imagens na base de treino:", len(df_treino_original))
print("Pacientes na base de treino:", df_treino_original["patient_id"].nunique())
print("Imagens na base de teste:", len(df_teste))
print("Pacientes na base de teste:", df_teste["patient_id"].nunique())
print("Classes no treino:", sorted(df_treino_original["label"].unique()))
print("Classes no teste:", sorted(df_teste["label"].unique()))
display(df_treino_original.head())

Imagens na base de treino: 593
Pacientes na base de treino: 20
Imagens na base de teste: 371
Pacientes na base de teste: 13
Classes no treino: ['0', '1', '2', '3']
Classes no teste: ['0', '1', '2', '3']


,image_path,filename,patient_id,label
0,C:\workspace\ufpr\IAA-2026\TRABALHO_DE_IAA011\...,01_HER2_2775.png,01,0
1,C:\workspace\ufpr\IAA-2026\TRABALHO_DE_IAA011\...,01_HER2_3372.png,01,0
2,C:\workspace\ufpr\IAA-2026\TRABALHO_DE_IAA011\...,01_HER2_3570.png,01,0
3,C:\workspace\ufpr\IAA-2026\TRABALHO_DE_IAA011\...,01_HER2_3973.png,01,0
4,C:\workspace\ufpr\IAA-2026\TRABALHO_DE_IAA011\...,01_HER2_4153.png,01,0


### Passo 3 — Divide os dados de treinamento em treinamento 80% e validação 20%
Foi dividido a base em cinco partes mantendo junto imagens do mesmo paciente

In [32]:
# Cria 5 divisões, mantendo as imagens de cada paciente juntas
divisor = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

indices_treino, indices_validacao = next(
    divisor.split(
        df_treino_original,
        df_treino_original["label"],
        groups=df_treino_original["patient_id"]
    )
)

df_treino = df_treino_original.iloc[indices_treino].copy()
df_validacao = df_treino_original.iloc[indices_validacao].copy()

# Garante que nenhum paciente aparece nos dois conjuntos.
assert set(df_treino["patient_id"]).isdisjoint(set(df_validacao["patient_id"]))

print("Divisão concluída!")
print("Imagens de treino:", len(df_treino))
print("Imagens de validação:", len(df_validacao))

Divisão concluída!
Imagens de treino: 476
Imagens de validação: 117


### Passo 4 — Função de extração LBP e Salvar CSV

LBP compara cada pixel com seus vizinhos para descrever a textura. Para cada imagem, calculamos o LBP em escala de cinza e contamos a frequência das 10 categorias. O histograma é normalizado para que seus valores somem 1.

In [5]:
def extrair_lbp(caminho_imagem):
    with Image.open(caminho_imagem) as imagem:
        imagem_cinza = np.asarray(imagem.convert("L"), dtype=np.uint8)

    lbp = local_binary_pattern(
        imagem_cinza, P=LBP_P, R=LBP_R, method=LBP_METHOD
    )
    histograma = np.bincount(
        lbp.astype(np.int32).ravel(), minlength=N_LBP_BINS
    )[:N_LBP_BINS].astype(np.float32)

    return histograma / histograma.sum()

Cada linha do CSV representa uma imagem; as primeiras colunas guardam os metadados e as 10 colunas seguintes guardam as características LBP.

In [6]:
def gerar_csv_lbp(df, nome_particao):
    linhas = []
    for registro in df.itertuples(index=False):
        vetor = extrair_lbp(registro.image_path)
        linhas.append([
            registro.image_path, registro.filename,
            registro.patient_id, registro.label, *vetor
        ])

    colunas = ["image_path", "filename", "patient_id", "label"]
    colunas += [f"lbp_{i}" for i in range(N_LBP_BINS)]
    resultado = pd.DataFrame(linhas, columns=colunas)
    resultado.to_csv(OUTPUT_DIR / f"features_lbp_{nome_particao}.csv", index=False)
    print(f"LBP {nome_particao}: {resultado.shape}")
    return resultado


df_lbp_treino = gerar_csv_lbp(df_treino, "treino")
df_lbp_validacao = gerar_csv_lbp(df_validacao, "validacao")
df_lbp_teste = gerar_csv_lbp(df_teste, "teste")
display(df_lbp_treino.head())

LBP treino: (476, 14)
LBP validacao: (117, 14)
LBP teste: (371, 14)


,image_path,filename,patient_id,label,lbp_0,lbp_1,lbp_2,lbp_3,lbp_4,lbp_5,lbp_6,lbp_7,lbp_8,lbp_9
0,C:\workspace\ufpr\IAA-2026\TRABALHO_DE_IAA011\...,01_HER2_2775.png,01,0,0.011856,0.031728,0.040160,0.154640,0.370224,0.215056,0.069376,0.033872,0.030752,0.042336
1,C:\workspace\ufpr\IAA-2026\TRABALHO_DE_IAA011\...,01_HER2_3372.png,01,0,0.015280,0.035856,0.044240,0.152272,0.345648,0.205952,0.072048,0.035904,0.044496,0.048304
2,C:\workspace\ufpr\IAA-2026\TRABALHO_DE_IAA011\...,01_HER2_3570.png,01,0,0.017600,0.039088,0.049232,0.154240,0.314112,0.204544,0.079920,0.039856,0.046352,0.055056
3,C:\workspace\ufpr\IAA-2026\TRABALHO_DE_IAA011\...,01_HER2_3973.png,01,0,0.013312,0.034544,0.037296,0.140656,0.332304,0.215808,0.071776,0.038288,0.068656,0.047360
4,C:\workspace\ufpr\IAA-2026\TRABALHO_DE_IAA011\...,01_HER2_4153.png,01,0,0.015408,0.037168,0.044640,0.152416,0.340352,0.208320,0.075760,0.037888,0.037472,0.050576


### Passo 5 — Carregar a VGG16 pré-treinada

Agora usamos a VGG16 com pesos do ImageNet. `include_top=True` mantém as camadas finais originais e escolhemos `fc2`, a penúltima camada, como saída. Ela gera um vetor de 4096 características por imagem. Os pesos ficam congelados porque estamos apenas extraindo características.

In [7]:
vgg16_base = VGG16(weights="imagenet", include_top=True, input_shape=(224, 224, 3))
vgg16_extrator = Model(
    inputs=vgg16_base.input,
    outputs=vgg16_base.get_layer("fc2").output
)
vgg16_extrator.trainable = False

VGG_SIZE = (224, 224)
VGG_BATCH_SIZE = 16
VGG_N_FEATURES = 4096

print("VGG16 pronta para extrair", VGG_N_FEATURES, "características por imagem.")

553467096/553467096 ━━━━━━━━━━━━━━━━━━━━ 18s 0us/step
VGG16 pronta para extrair 4096 características por imagem.


### Passo 6 — Preparar imagens e extrair características VGG16

A VGG16 espera imagens RGB de 224 × 224 pixels. A função abaixo redimensiona e aplica o pré-processamento esperado pelos pesos do ImageNet.

In [8]:
def preparar_imagem_vgg16(caminho_imagem):
    with Image.open(caminho_imagem) as imagem:
        imagem = imagem.convert("RGB").resize(VGG_SIZE)
        imagem = np.asarray(imagem, dtype=np.float32)
    return preprocess_input(imagem)


def gerar_csv_vgg16(df, nome_particao):
    lotes = []
    for inicio in range(0, len(df), VGG_BATCH_SIZE):
        lote_df = df.iloc[inicio:inicio + VGG_BATCH_SIZE]
        imagens = np.stack([
            preparar_imagem_vgg16(caminho)
            for caminho in lote_df["image_path"]
        ])
        lotes.append(vgg16_extrator.predict(imagens, verbose=0))
        print(f"VGG16 {nome_particao}: {min(inicio + VGG_BATCH_SIZE, len(df))}/{len(df)}")

    caracteristicas = np.concatenate(lotes, axis=0)
    colunas = [f"vgg16_{i}" for i in range(VGG_N_FEATURES)]
    df_caracteristicas = pd.DataFrame(caracteristicas, columns=colunas)
    metadados = df[["image_path", "filename", "patient_id", "label"]].reset_index(drop=True)
    resultado = pd.concat([metadados, df_caracteristicas], axis=1)
    resultado.to_csv(
        OUTPUT_DIR / f"features_vgg16_{nome_particao}.csv",
        index=False, float_format="%.7g"
    )
    print(f"CSV VGG16 {nome_particao}: {resultado.shape}")
    return resultado

In [9]:
df_vgg16_treino = gerar_csv_vgg16(df_treino, "treino")
df_vgg16_validacao = gerar_csv_vgg16(df_validacao, "validacao")
df_vgg16_teste = gerar_csv_vgg16(df_teste, "teste")

print("\nArquivos gerados:")
for arquivo in sorted(OUTPUT_DIR.glob("*.csv")):
    print(arquivo.name)

VGG16 treino: 16/476
VGG16 treino: 32/476
VGG16 treino: 48/476
VGG16 treino: 64/476
VGG16 treino: 80/476
VGG16 treino: 96/476
VGG16 treino: 112/476
VGG16 treino: 128/476
VGG16 treino: 144/476
VGG16 treino: 160/476
VGG16 treino: 176/476
VGG16 treino: 192/476
VGG16 treino: 208/476
VGG16 treino: 224/476
VGG16 treino: 240/476
VGG16 treino: 256/476
VGG16 treino: 272/476
VGG16 treino: 288/476
VGG16 treino: 304/476
VGG16 treino: 320/476
VGG16 treino: 336/476
VGG16 treino: 352/476
VGG16 treino: 368/476
VGG16 treino: 384/476
VGG16 treino: 400/476
VGG16 treino: 416/476
VGG16 treino: 432/476
VGG16 treino: 448/476
VGG16 treino: 464/476
VGG16 treino: 476/476
CSV VGG16 treino: (476, 4100)
VGG16 validacao: 16/117
VGG16 validacao: 32/117
VGG16 validacao: 48/117
VGG16 validacao: 64/117
VGG16 validacao: 80/117
VGG16 validacao: 96/117
VGG16 validacao: 112/117
VGG16 validacao: 117/117
CSV VGG16 validacao: (117, 4100)
VGG16 teste: 16/371
VGG16 teste: 32/371
VGG16 teste: 48/371
VGG16 teste: 64/371
VGG16 tes

### Passo 7 — Ler os CSVs e reunir a base de treinamento

A separação por paciente do passo 3 é mantida. Como a tarefa 4 pede todas as imagens da base original para treinar os classificadores, reunimos treino e validação de cada extrator. O teste não participa do treinamento.

Os modelos terão configurações fixas, sem busca de parâmetros. Se os CSVs já existem, podemos executar os passos 1–3 e continuar daqui, sem repetir a extração dos passos 4–6.

In [33]:
# Lê os seis arquivos, preservando os IDs dos pacientes e os rótulos como texto.
def ler_csv(nome):
    return pd.read_csv(OUTPUT_DIR / nome, dtype={"patient_id": str, "label": str})

df_lbp_treino = ler_csv("features_lbp_treino.csv")
df_lbp_validacao = ler_csv("features_lbp_validacao.csv")
df_lbp_teste = ler_csv("features_lbp_teste.csv")

df_vgg16_treino = ler_csv("features_vgg16_treino.csv")
df_vgg16_validacao = ler_csv("features_vgg16_validacao.csv")
df_vgg16_teste = ler_csv("features_vgg16_teste.csv")

lbp_completo = pd.concat([df_lbp_treino, df_lbp_validacao], ignore_index=True)
vgg16_completo = pd.concat([df_vgg16_treino, df_vgg16_validacao], ignore_index=True)

# Nenhum paciente da base de treinamento pode aparecer no teste.
assert set(lbp_completo["patient_id"]).isdisjoint(df_lbp_teste["patient_id"])
assert set(vgg16_completo["patient_id"]).isdisjoint(df_vgg16_teste["patient_id"])

print("Base completa de treino:", len(lbp_completo), "imagens")
print("Base de teste:", len(df_lbp_teste), "imagens")

RESULTADOS_DIR = OUTPUT_DIR / "resultados_simples"
RESULTADOS_DIR.mkdir(exist_ok=True)

Base completa de treino: 593 imagens
Base de teste: 371 imagens


### Passo 8 — Separar características (X) e classes (y)

X contém somente as colunas do extrator. Caminhos, nomes, pacientes e rótulos ficam fora da entrada do modelo. As classes internas `0, 1, 2, 3` correspondem a `0, 1+, 2+, 3+`.

In [34]:
X_lbp_treino = lbp_completo.filter(regex="^lbp_")
y_lbp_treino = lbp_completo["label"]
X_lbp_teste = df_lbp_teste.filter(regex="^lbp_")
y_lbp_teste = df_lbp_teste["label"]

X_vgg16_treino = vgg16_completo.filter(regex="^vgg16_")
y_vgg16_treino = vgg16_completo["label"]
X_vgg16_teste = df_vgg16_teste.filter(regex="^vgg16_")
y_vgg16_teste = df_vgg16_teste["label"]

print("LBP — treino:", X_lbp_treino.shape, "| teste:", X_lbp_teste.shape)
print("VGG16 — treino:", X_vgg16_treino.shape, "| teste:", X_vgg16_teste.shape)

LBP — treino: (593, 10) | teste: (371, 10)
VGG16 — treino: (593, 4096) | teste: (371, 4096)


### Passo 9 — Definir os três classificadores

Random Forest combina árvores de decisão; SVM usa kernel RBF; a RNA é uma rede multicamada (`MLPClassifier`) com uma camada oculta de 64 neurônios.

SVM e RNA usam padronização dentro de um pipeline. O normalizador é ajustado somente no treino e a mesma transformação é aplicada ao teste. A função cria modelos novos para cada extrator.

In [35]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix

def criar_modelos():
    return {
        "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
        "SVM": make_pipeline(StandardScaler(), SVC(kernel="rbf", C=1)),
        "RNA": make_pipeline(
            StandardScaler(),
            MLPClassifier(hidden_layer_sizes=(64,), solver="lbfgs",
                          alpha=0.01, max_iter=2000, random_state=42)
        ),
    }

### Passo 10 — Calcular as métricas pela matriz de confusão

As linhas representam classes reais e as colunas representam classes preditas. Para cada classe, ela é considerada positiva e as demais são negativas.

- Sensibilidade: TP / (TP + FN).
- Especificidade: TN / (TN + FP).
- F1: 2TP / (2TP + FP + FN).

Se o denominador for zero, a métrica recebe zero. A média macro é a média das quatro classes, com o mesmo peso para cada uma.

In [36]:
CLASSES = ["0", "1", "2", "3"]
NOMES_CLASSES = ["0", "1+", "2+", "3+"]

def calcular_metricas(matriz):
    linhas = []
    for i, classe in enumerate(NOMES_CLASSES):
        tp = matriz[i, i]
        fn = matriz[i, :].sum() - tp
        fp = matriz[:, i].sum() - tp
        tn = matriz.sum() - tp - fn - fp

        sensibilidade = tp / (tp + fn) if tp + fn > 0 else 0
        especificidade = tn / (tn + fp) if tn + fp > 0 else 0
        f1 = 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn > 0 else 0

        linhas.append({
            "classe": classe,
            "sensibilidade": sensibilidade,
            "especificidade": especificidade,
            "f1": f1,
        })
    return pd.DataFrame(linhas)

### Passo 11 — Treinar e avaliar os modelos

A função abaixo executa a mesma sequência para cada classificador: treinar, prever o teste, mostrar a matriz e as métricas por classe e guardar as médias macro. As tabelas são salvas em CSV para a entrega.

In [37]:
def treinar_e_avaliar(extrator, X_treino, y_treino, X_teste, y_teste):
    resultados = []
    for nome, modelo in criar_modelos().items():
        print(f"Treinando {extrator} + {nome}...", flush=True)
        modelo.fit(X_treino, y_treino)
        predicoes = modelo.predict(X_teste)

        matriz = confusion_matrix(y_teste, predicoes, labels=CLASSES)
        tabela_matriz = pd.DataFrame(
            matriz, index=NOMES_CLASSES, columns=NOMES_CLASSES
        ).rename_axis(index="Real", columns="Predita")
        metricas = calcular_metricas(matriz)

        print("Matriz de confusão:")
        display(tabela_matriz)
        print("Métricas por classe:")
        display(metricas.round(4))

        arquivo = f"{extrator}_{nome.replace(' ', '_')}"
        tabela_matriz.to_csv(RESULTADOS_DIR / f"matriz_{arquivo}.csv")
        metricas.to_csv(RESULTADOS_DIR / f"metricas_{arquivo}.csv", index=False)
        pd.DataFrame({
            "classe_real": y_teste.to_numpy(), "classe_predita": predicoes
        }).to_csv(RESULTADOS_DIR / f"predicoes_{arquivo}.csv", index=False)

        resultados.append({
            "extrator": extrator, "modelo": nome,
            "sensibilidade_macro": metricas["sensibilidade"].mean(),
            "especificidade_macro": metricas["especificidade"].mean(),
            "f1_macro": metricas["f1"].mean(),
        })
    return pd.DataFrame(resultados)

### Passo 12 — Executar os três modelos com LBP

In [38]:
resultados_lbp = treinar_e_avaliar(
    "LBP", X_lbp_treino, y_lbp_treino, X_lbp_teste, y_lbp_teste
)
display(resultados_lbp.round(4))

Treinando LBP + Random Forest...
Matriz de confusão:


Predita,0,1+,2+,3+
Real,,,,
0,48,29,23,1
1+,20,46,24,0
2+,14,30,42,4
3+,1,0,9,80


Métricas por classe:


,classe,sensibilidade,especificidade,f1
0,0,0.4752,0.8704,0.5217
1,1+,0.5111,0.7900,0.4718
2,2+,0.4667,0.8007,0.4468
3,3+,0.8889,0.9822,0.9143


Treinando LBP + SVM...
Matriz de confusão:


Predita,0,1+,2+,3+
Real,,,,
0,49,29,23,0
1+,17,48,25,0
2+,14,31,45,0
3+,1,0,9,80


... (4 outputs left out)

Predita,0,1+,2+,3+
Real,,,,
0,49,27,25,0
1+,21,38,31,0
2+,15,23,50,2
3+,1,0,16,73


Métricas por classe:


,classe,sensibilidade,especificidade,f1
0,0,0.4851,0.8630,0.5241
1,1+,0.4222,0.8221,0.4270
2,2+,0.5556,0.7438,0.4717
3,3+,0.8111,0.9929,0.8848


,extrator,modelo,sensibilidade_macro,especificidade_macro,f1_macro
0,LBP,Random Forest,0.5855,0.8608,0.5887
1,LBP,SVM,0.6018,0.8663,0.6083
2,LBP,RNA,0.5685,0.8554,0.5769


### Passo 13 — Executar os três modelos com VGG16

In [39]:
resultados_vgg16 = treinar_e_avaliar(
    "VGG16", X_vgg16_treino, y_vgg16_treino, X_vgg16_teste, y_vgg16_teste
)
display(resultados_vgg16.round(4))

Treinando VGG16 + Random Forest...
Matriz de confusão:


Predita,0,1+,2+,3+
Real,,,,
0,97,4,0,0
1+,14,41,35,0
2+,0,17,71,2
3+,0,0,1,89


Métricas por classe:


,classe,sensibilidade,especificidade,f1
0,0,0.9604,0.9481,0.9151
1,1+,0.4556,0.9253,0.5395
2,2+,0.7889,0.8719,0.7208
3,3+,0.9889,0.9929,0.9834


Treinando VGG16 + SVM...
Matriz de confusão:


Predita,0,1+,2+,3+
Real,,,,
0,94,3,2,2
1+,8,42,33,7
2+,0,14,68,8
3+,0,0,0,90


... (3 outputs left out)

Predita,0,1+,2+,3+
Real,,,,
0,91,7,3,0
1+,8,54,28,0
2+,0,17,71,2
3+,0,0,1,89


Métricas por classe:


,classe,sensibilidade,especificidade,f1
0,0,0.9010,0.9704,0.9100
1,1+,0.6000,0.9146,0.6429
2,2+,0.7889,0.8861,0.7358
3,3+,0.9889,0.9929,0.9834


,extrator,modelo,sensibilidade_macro,especificidade_macro,f1_macro
0,VGG16,Random Forest,0.7984,0.9345,0.7897
1,VGG16,SVM,0.7882,0.9312,0.7771
2,VGG16,RNA,0.8197,0.9410,0.8180


### Passo 14 — Comparar e indicar o melhor resultado

Usamos **F1 macro** como métrica principal porque dá o mesmo peso às quatro classes e considera falsos positivos e falsos negativos. A tabela é ordenada por essa métrica.

O maior resultado observado no teste responde à comparação do exercício; não é utilizado para ajustar parâmetros. Em empate, as combinações empatadas são apresentadas. Os resultados se referem às bases locais de amostra e não demonstram superioridade estatística.

In [40]:
comparacao = pd.concat([resultados_lbp, resultados_vgg16], ignore_index=True)
comparacao = comparacao.sort_values("f1_macro", ascending=False).reset_index(drop=True)
display(comparacao.round(4))
comparacao.to_csv(RESULTADOS_DIR / "comparacao_modelos.csv", index=False)

melhor_f1 = comparacao["f1_macro"].max()
melhores = comparacao[comparacao["f1_macro"] == melhor_f1]

print("Melhor resultado pelo F1 macro:")
for _, resultado in melhores.iterrows():
    print(f"{resultado['extrator']} + {resultado['modelo']}: {resultado['f1_macro']:.4f}")
print("Justificativa: maior F1 macro, atribuindo o mesmo peso às quatro classes.")
print("CSVs salvos em:", RESULTADOS_DIR)

,extrator,modelo,sensibilidade_macro,especificidade_macro,f1_macro
0,VGG16,RNA,0.8197,0.9410,0.8180
1,VGG16,Random Forest,0.7984,0.9345,0.7897
2,VGG16,SVM,0.7882,0.9312,0.7771
3,LBP,SVM,0.6018,0.8663,0.6083
4,LBP,Random Forest,0.5855,0.8608,0.5887
5,LBP,RNA,0.5685,0.8554,0.5769


Melhor resultado pelo F1 macro:
VGG16 + RNA: 0.8180
Justificativa: maior F1 macro, atribuindo o mesmo peso às quatro classes.
CSVs salvos em: C:\workspace\ufpr\IAA-2026\TRABALHO_DE_IAA011\Trabalho Imagens - IAA\saidas_exercicio1\resultados_simples
